# Entrenar un segmentador de regiones de layout (one-shot) — Etiquetado de Manuscritos

Este notebook implementa la receta de **De Nardin et al. (WACV 2024)** — *A One-Shot Learning Approach to Document Layout Segmentation of Ancient Arabic Manuscripts* — para distinguir **texto principal / nota al margen / fondo**, usando como ground truth las 33 `LayoutRegion` ya etiquetadas manualmente (exportadas con `backend/scripts/export_layout_masks.py`).

Firma, Sello/Timbre, Encabezado, Inscripción extra y Decoración quedan fuera de este modelo (siguen etiquetándose a mano) — alcance acordado en el vault (`etiquetado-manuscritos.md`).

**Antes de correr este notebook:**
1. En tu máquina local: `cd backend && python -m scripts.export_layout_masks --project-id 1`
2. Comprimí **solo** `images/` y `masks/` (no hace falta `masks_preview/`) dentro de `colab/dataset_layout/` en un `dataset_layout.zip` y subilo en la celda de abajo.

**Salida:** un archivo `layout_segmenter.onnx` descargable.

## 0. Setup

In [ ]:
!pip install -q scikit-image onnx onnxruntime onnxscript
import torch, torchvision
print('torch', torch.__version__, '| cuda disponible:', torch.cuda.is_available())

## 1. Subir y descomprimir el dataset exportado

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # selecciona dataset.zip
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('dataset')

images_dir = 'dataset/images'
masks_dir = 'dataset/masks'
ids = sorted(f.split('.')[0] for f in os.listdir(images_dir))
print(f'{len(ids)} páginas etiquetadas encontradas:', ids)
assert len(ids) >= 1, 'Exporta al menos 1 página con backend/scripts/export_word_masks.py'

## 2. Dataset + Generación Dinámica de Instancias

Reproduce el módulo del paper: redimensiona cada página a un tamaño fijo, la divide en un
grid de parches base sin solape, y añade `k` recortes aleatorios adicionales **por época**
(no fijos entre épocas) como augmentation — es la pieza que hace viable entrenar con 1-3
imágenes. Valores por defecto = los del paper (imagen 1344×2016, parches 672×672, k=12).

In [ ]:
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

RESIZE_TO = (1344, 2016)   # (alto, ancho) — apaisado, porque los bifolios de este corpus son mas anchos que altos
PATCH_SIZE = 672
RANDOM_CROPS_PER_EPOCH = 12

def load_pair(doc_id):
    img = Image.open(f'{images_dir}/{doc_id}.png').convert('RGB').resize((RESIZE_TO[1], RESIZE_TO[0]))
    mask = Image.open(f'{masks_dir}/{doc_id}.png').convert('L').resize((RESIZE_TO[1], RESIZE_TO[0]), Image.NEAREST)
    return np.array(img), np.array(mask).astype(np.int64)   # ya son indices de clase (0/1/2), no hace falta binarizar

def grid_patches(img, mask, patch=PATCH_SIZE):
    h, w = mask.shape
    out = []
    for y in range(0, h - patch + 1, patch):
        for x in range(0, w - patch + 1, patch):
            out.append((img[y:y+patch, x:x+patch], mask[y:y+patch, x:x+patch]))
    return out

def random_crops(img, mask, n=RANDOM_CROPS_PER_EPOCH, patch=PATCH_SIZE):
    h, w = mask.shape
    out = []
    for _ in range(n):
        y = random.randint(0, h - patch)
        x = random.randint(0, w - patch)
        out.append((img[y:y+patch, x:x+patch], mask[y:y+patch, x:x+patch]))
    return out

class LayoutPatchDataset(Dataset):
    """Regenera los recortes aleatorios en cada epoca (Dynamic Instance Generation)."""
    def __init__(self, doc_ids):
        self.pairs = [load_pair(d) for d in doc_ids]
        self.regenerate()

    def regenerate(self):
        patches = []
        for img, mask in self.pairs:
            patches += grid_patches(img, mask)
            patches += random_crops(img, mask)
        self.patches = patches

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, i):
        img, mask = self.patches[i]
        img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        mask_t = torch.from_numpy(mask).long()
        return img_t, mask_t

## 3. Split train/val

Con muy pocas páginas, usa la(s) misma(s) página(s) para train y valida contra recortes
distintos, o si tienes >=2 páginas, deja 1 fuera para validación real. Ajusta `val_ids`
manualmente según cuántas páginas hayas etiquetado.

In [ ]:
val_ids = ids[-1:] if len(ids) > 1 else ids
train_ids = [i for i in ids if i not in val_ids] or ids
print('train:', train_ids, '| val:', val_ids)

train_ds = LayoutPatchDataset(train_ids)
val_ds = LayoutPatchDataset(val_ids)
print(f'{len(train_ds)} parches de train, {len(val_ds)} parches de val (por época)')

## 4. Modelo: DeepLabv3+ (backbone ResNet50 preentrenado)

El paper usa Aligned XCeption; aquí usamos `deeplabv3_resnet50` de torchvision (pesos
preentrenados en COCO) por disponibilidad directa en Colab — el propio paper reporta que
el backbone no es la pieza crítica del método (sí lo son Dynamic Instance Generation +
Sauvola), así que este cambio es razonable. 2 clases: fondo / palabra.

In [ ]:
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = deeplabv3_resnet50(weights=DeepLabV3_ResNet50_Weights.DEFAULT)
model.classifier[4] = torch.nn.Conv2d(256, 3, kernel_size=1)          # fondo / texto principal / nota al margen
model.aux_classifier[4] = torch.nn.Conv2d(256, 3, kernel_size=1)
model = model.to(device)
print('Modelo listo en', device)

## 5. Pérdida: Jaccard + Dice (igual que el paper)

In [ ]:
import torch.nn.functional as F

def jaccard_dice_loss(logits, target, num_classes=3, eps=1e-6):
    probs = F.softmax(logits, dim=1)                                        # (B, C, H, W)
    target_onehot = F.one_hot(target, num_classes).permute(0, 3, 1, 2).float()  # (B, C, H, W)

    dims = (0, 2, 3)  # suma sobre batch + espacio, deja viva la dimension de clase
    inter = (probs * target_onehot).sum(dim=dims)
    union = probs.sum(dim=dims) + target_onehot.sum(dim=dims) - inter

    jaccard = 1 - (inter + eps) / (union + eps)
    dice = 1 - (2 * inter + eps) / (probs.sum(dim=dims) + target_onehot.sum(dim=dims) + eps)

    return (jaccard.mean() + dice.mean()) / 2   # promedio final sobre las 3 clases

## 6. Entrenamiento (Adam, early stopping — hiperparámetros del paper)

`lr=1e-3`, `weight_decay=1e-5`, máx. 200 épocas, corta si val no mejora en 20 épocas
consecutivas (con un margen/buffer de 50 épocas antes de poder cortar).

In [ ]:
from torch.utils.data import DataLoader

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
MAX_EPOCHS, PATIENCE, BUFFER = 200, 20, 50
best_val, epochs_no_improve, best_state = float('inf'), 0, None

for epoch in range(MAX_EPOCHS):
    train_ds.regenerate()  # nuevos recortes aleatorios cada época
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=4)

    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)['out']
        loss = jaccard_dice_loss(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)['out']
            val_loss += jaccard_dice_loss(out, y).item() * x.size(0)
    val_loss /= len(val_ds)

    print(f'epoch {epoch+1:3d} | train {train_loss:.4f} | val {val_loss:.4f}')

    if val_loss < best_val:
        best_val, epochs_no_improve = val_loss, 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epoch > BUFFER and epochs_no_improve >= PATIENCE:
            print(f'Early stopping en epoch {epoch+1} (sin mejora en {PATIENCE} épocas)')
            break

model.load_state_dict(best_state)
print('Mejor val loss:', best_val)

## 7. Refinamiento post-inferencia con Sauvola

Igual que el paper: `window_size=31`, `k=0.2` — calibra estos dos valores empíricamente
sobre tus propias páginas si los bordes de las cajas de palabra salen muy gruesos/delgados
(el propio paper señala esto como su principal limitación).

In [ ]:
from skimage.filters import threshold_sauvola
from skimage.color import rgb2gray

def predict_mask(model, img_rgb_uint8):
    x = torch.from_numpy(img_rgb_uint8).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    with torch.no_grad():
        logits = model(x.to(device))['out']
        coarse = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
    return coarse

def refine_with_sauvola(img_rgb_uint8, coarse_mask, window_size=31, k=0.2):
    gray = rgb2gray(img_rgb_uint8)
    thresh = threshold_sauvola(gray, window_size=window_size, k=k)
    ink = (gray < thresh)
    refined = coarse_mask.copy()
    refined[(coarse_mask > 0) & ~ink] = 0   # dentro de una region predicha, lo que no es tinta real vuelve a fondo
    return refined

## 8. Visualizar sobre una página de validación

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

CMAP = ListedColormap(['black', '#3B82F6', '#F59E0B'])  # fondo / texto principal / nota al margen

sample_id = val_ids[0]
img, gt = load_pair(sample_id)
coarse = predict_mask(model, img)
refined = refine_with_sauvola(img, coarse)

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
for ax, im, title in zip(axes, [img, gt, coarse, refined],
                          ['Original', 'GT (manual)', 'Predicción gruesa', 'Refinado (Sauvola)']):
    if im.ndim == 2:
        ax.imshow(im, cmap=CMAP, vmin=0, vmax=2)
    else:
        ax.imshow(im)
    ax.set_title(title)
    ax.axis('off')
plt.show()

## 9. Exportar a ONNX (para inferencia en el backend sin depender de PyTorch)

El backend actual (`segmentation_service.py`) solo usa OpenCV. Para no añadir PyTorch
como dependencia de producción, exporta a ONNX y usa `onnxruntime` (CPU) en su lugar —
mucho más liviano de instalar/mantener en el servidor.

In [ ]:
model.eval()
dummy = torch.randn(1, 3, PATCH_SIZE, PATCH_SIZE, device=device)
torch.onnx.export(
    model, dummy, 'layout_segmenter.onnx',
    input_names=['image'], output_names=['logits'],
    dynamic_axes={'image': {0: 'batch', 2: 'height', 3: 'width'}, 'logits': {0: 'batch', 2: 'height', 3: 'width'}},
    opset_version=17,
    dynamo=False,
)
import os
print('Tamaño del .onnx:', os.path.getsize('layout_segmenter.onnx') / 1e6, 'MB')
files.download('layout_segmenter.onnx')
print('Descarga layout_segmenter.onnx y colócalo en backend/app/ml/layout_segmenter.onnx')

## Próximo paso en el backend (no incluido en este notebook)

1. `onnxruntime` en `backend/requirements.txt`.
2. Nueva función en `backend/app/services/layout_ml_service.py`: carga `layout_segmenter.onnx`,
   corre inferencia por parches (mismo grid `PATCH_SIZE=672`), aplica Sauvola, y separa la
   máscara de 3 clases en bboxes por región con `cv2.findContours` (uno por cada clase
   1/2), mismo formato de salida que el heurístico (`x/y/width/height/region_type/confidence`).
3. Endpoint `"ml"` en `backend/app/routers/layout.py`, patrón preview → apply.
4. Selector de método en `DLAPanel.tsx` (frontend), igual que ya existe en `LabelingPanel.tsx`.